## Sample notebook to demo `H3_LatLngToCell` function.

```shell
uv pip install geopandas xyzservices folium matplotlib mapclassify
```

In [1]:
import geofunctions as S
import geopandas as gp
import xyzservices.providers as xyz
from pyspark.sql import functions as F
from pyspark.sql import types as T

The continental United States extends roughly from approximately 25°N to 49°N in latitude and from 67°W to 125°W in longitude. 

In [2]:
xmin, xmax = (-125.0, -67.0)
ymin, ymax = (25.0, 49.0)
xdel = xmax - xmin
ydel = ymax - ymin

### Generate sample lat/lon values.

In [3]:
df = (
    spark.range(100_000)
    .withColumn("lat", F.rand() * ydel + ymin)
    .withColumn("lng", F.rand() * xdel + xmin)
)

### Aggregate the data and convert the cell to a geometry.

In [4]:
pdf = (
    df.select(S.h3_latlng_to_cell("lat", "lng", 4))
    .groupBy("cell")
    .count()
    .orderBy("count")
    .select(
        S.h3_cell_to_boundary().alias("geometry"),
        F.least(F.col("count"), F.lit(30.0)).alias("pop"),
    )
    .toPandas()
)

### Convert Pandas DF to GeoPandas DF.

In [5]:
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="EPSG:4326")
gdf.sindex;

In [6]:
gdf.explore(
    "pop",
    cmap="coolwarm",
    vmin=0.0,
    vmax=30.0,
    tiles=xyz.Esri.WorldGrayCanvas,
)